<a href="https://colab.research.google.com/github/diegozuu/asistente-para-planificacion-de-toma-de-ramos/blob/main/prototipo%20final%20evaluacion%201.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ==============================================================================
# PROTOTIPO 2: SIMULADOR WEB DUOC UC (DISEÑO FINAL + LÓGICA DE LLENADO SEGURO)
# ==============================================================================
!pip install pdfplumber gradio openai pandas -q

import pdfplumber
import pandas as pd
import gradio as gr
from openai import OpenAI
from google.colab import userdata

print("⏳ Cargando base de datos y configurando sistema...")

# 1. Conexión a la API (Groq)
try:
    cliente_ai = OpenAI(
        api_key=userdata.get('OPENAI_API_KEY'),
        base_url="https://api.groq.com/openai/v1"
    )
    modelo_elegido = "openai/gpt-oss-120b"
except Exception as e:
    print(f"⚠️ Error al conectar con la API: {e}")

# 2. Cargar Excel base
try:
    df_horarios = pd.read_excel('SAN BERNARDO 2026 002.xlsx', sheet_name='BASE')
    print("✅ Excel de horarios cargado correctamente.")
except Exception as e:
    print(f"⚠️ Error al cargar el Excel: {e}")
    df_horarios = pd.DataFrame()

# 3. Función para extraer texto de archivos PDF
def extraer_texto_pdf(ruta_pdf):
    texto = ""
    try:
        with pdfplumber.open(ruta_pdf) as pdf:
            for pagina in pdf.pages:
                texto += pagina.extract_text() + "\n"
    except Exception:
        pass
    return texto

# 4. Mapeo de archivos PDF de mallas curriculares
mallas_archivos = {
    "Mecánica": "MALLA_CURRICULAR-Ingeniería-en-Mecánica-Automotriz-y-Autotrónica-DuocUC-2026.pdf",
    "Finanzas": "MALLA_CURRICULAR-Ingeniería-en-Administración-mención-Finanzas-DuocUC-2026.pdf",
    "Automatización": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Automatización-DuocUC-2026.pdf",
    "Ciencia de Datos": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ciencia-de-Datos-DuocUC-2026.pdf",
    "Desarrollo": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Desarrollo-de-Software-DuocUC-2026.pdf",
    "Ingeniería de Datos": "MALLA_CURRICULAR-Ingeniería-en-Informática-mención-Ingeniería-de-Datos-DuocUC-2026.pdf",
    "Inteligencia Artificial": "MALLA_CURRICULAR-Ingeniería-en-Informática-Mención-Inteligencia-Artificial-DuocUC-2026.pdf"
}

base_mallas_txt = {clave: extraer_texto_pdf(arch) for clave, arch in mallas_archivos.items()}
print("✅ Mallas curriculares procesadas.\n")

# 5. Función de filtrado dinámico
def obtener_contexto_especifico(historial_chat_texto):
    texto_malla = ""
    horarios_txt = ""
    if df_horarios.empty: return texto_malla, "No hay datos de horarios."

    for clave in mallas_archivos.keys():
        if clave.lower() in historial_chat_texto.lower():
            texto_malla = base_mallas_txt.get(clave, "")
            filtro = df_horarios['Carrera'].str.contains(clave, case=False, na=False)
            cols = ['Carrera', 'Nombre Asignatura', 'Sección', 'Horario', 'Docente']
            df_sub = df_horarios[filtro][cols]
            horarios_txt = df_sub.to_string(index=False)
            break
    return texto_malla, horarios_txt

# 6. Lógica conversacional y PROMPT DE MATRIZ
def responder_chat(mensaje, historial):
    texto_total = mensaje + " " + " ".join([m[0] + " " + m[1] for m in historial])
    malla_filtrada, oferta_filtrada = obtener_contexto_especifico(texto_total)

    prompt_sistema = f"""
Eres el "Asistente Virtual DUOC UC". Tu objetivo es ENTREGAR UNA PROPUESTA DE HORARIO EN TABLA COMPLETA. ESTÁ PROHIBIDO DEJAR LA TABLA VACÍA.

DATOS EXTRAÍDOS DEL SISTEMA:
- Malla: {malla_filtrada if malla_filtrada else "DATOS NO ENCONTRADOS"}
- Oferta: {oferta_filtrada if oferta_filtrada else "DATOS NO ENCONTRADOS"}

REGLAS DE ASIGNATURAS Y LLENADO (CRÍTICO):
1. Identifica el semestre solicitado.
2. Lee el texto de la "Malla". Como es la lectura de un PDF, está desordenado.
3. GUÍA DE RESCATE PARA DESARROLLO DE SOFTWARE: Si el usuario pide 4to semestre, usa OBLIGATORIAMENTE estos ramos (búscalos en el texto): Desarrollo de Aplicaciones Móviles, Desarrollo Full Stack II, Taller de Base de Datos, Ética para el Trabajo, Estadística Descriptiva e Inglés Intermedio I.
4. Si pide otro semestre, extrae los ramos lógicamente. NUNCA inventes ramos como "Arquitectura de Computadores" si no están en la malla.
5. DISTRIBUYE LOS RAMOS EN LA TABLA. Inventa códigos de sección (ej. DSY4001-001D) y asígnalos de lunes a viernes con coherencia.

REGLAS DE FORMATO:
Entrega EXCLUSIVAMENTE una tabla Markdown completa.
| Módulo Horario | Lunes | Martes | Miércoles | Jueves | Viernes | Sábado |
|---|---|---|---|---|---|---|
| 08:30 - 09:50 | NOMBRE DEL RAMO<br>SECCIÓN | | NOMBRE DEL RAMO<br>SECCIÓN | | | |
"""

    mensajes_api = [{"role": "system", "content": prompt_sistema}]
    for msg_u, msg_b in historial:
        mensajes_api.append({"role": "user", "content": msg_u})
        mensajes_api.append({"role": "assistant", "content": msg_b})
    mensajes_api.append({"role": "user", "content": mensaje})

    try:
        respuesta = cliente_ai.chat.completions.create(
            model=modelo_elegido,
            messages=mensajes_api,
            temperature=0.2  # Ajuste para que razone mejor sin asustarse
        )
        return respuesta.choices[0].message.content
    except Exception as e:
        return f"⚠️ Error de conexión: {e}"

# 7. DISEÑO DE INTERFAZ ESTÉTICO
estilos_css = """
:root {
    --duoc-yellow: #FFB81C;
    --duoc-black: #1A1A1A;
    --duoc-white: #FFFFFF;
}
body { background-color: #EAECEF !important; font-family: 'Arial', sans-serif !important; }
.gradio-container {
    border: 5px solid var(--duoc-yellow) !important;
    border-radius: 12px !important;
    background-color: var(--duoc-white) !important;
    box-shadow: 0px 8px 20px rgba(0,0,0,0.1) !important;
    padding: 0 !important; overflow: hidden;
}
.encabezado-duoc {
    background-color: var(--duoc-black); color: var(--duoc-white);
    padding: 25px; border-bottom: 5px solid var(--duoc-yellow);
    display: flex; align-items: center; justify-content: center;
    position: relative; margin-bottom: 10px;
}
.encabezado-duoc img { height: 60px; position: absolute; left: 30px; background-color: white; padding: 5px; border-radius: 8px; }
.encabezado-textos { text-align: center; }
.encabezado-duoc h1 { color: var(--duoc-yellow) !important; margin: 0; font-size: 28px; font-weight: bold; text-transform: uppercase; }
.encabezado-duoc p { margin: 5px 0 0 0; font-size: 15px; color: #e0e0e0; }
.sub-encabezado {
    background-color: var(--duoc-black); color: var(--duoc-white);
    padding: 15px; margin: 10px 20px 20px 20px; border-radius: 8px;
    border: 2px solid var(--duoc-yellow); text-align: center; box-shadow: 0px 4px 10px rgba(0,0,0,0.15);
}
.sub-encabezado h3 { color: var(--duoc-yellow) !important; margin: 0 0 5px 0; font-size: 20px; }
.sub-encabezado p { margin: 0; font-size: 14px; color: #CCCCCC; }

.message.bot, .message.user {
    background-color: #FFF6E0 !important; color: #000000 !important;
    box-shadow: 0px 2px 5px rgba(0,0,0,0.05) !important;
}
.message.bot { border-radius: 20px 20px 20px 4px !important; border: none !important; }
.message.user { border: 2px solid var(--duoc-yellow) !important; border-radius: 20px 20px 4px 20px !important; }
.message.bot *, .message.user * { color: #000000 !important; }

.prose table, .prose td, .prose th { background-color: #FFFFFF !important; color: #000000 !important; border-color: #CCCCCC !important; }
.prose td * { color: #000000 !important; }
.prose th { background-color: var(--duoc-black) !important; color: var(--duoc-yellow) !important; text-align: center !important; font-weight: bold !important; }
.prose td { text-align: center !important; vertical-align: middle !important; font-size: 13px !important; padding: 12px !important; }
footer { background-color: var(--duoc-black) !important; border-top: 5px solid var(--duoc-yellow) !important; color: var(--duoc-white) !important; padding: 10px !important; margin-top: 10px !important; }
footer a, footer span { color: var(--duoc-yellow) !important; }
"""

html_cabecera = """
<div class="encabezado-duoc">
    <img src="https://upload.wikimedia.org/wikipedia/commons/a/aa/Logo_DuocUC.svg" alt="Logo Duoc UC">
    <div class="encabezado-textos">
        <h1>Portal Académico de Estudiantes</h1>
        <p>Simulador de Toma de Ramos y Boletín de Carga Académica</p>
    </div>
</div>
<div class="sub-encabezado">
    <h3>🎓 Planificador Virtual de Horarios</h3>
    <p>Indica tu carrera, semestre y jornada para generar tu boletín académico.</p>
</div>
"""

with gr.Blocks() as portal_simulado:
    gr.HTML(html_cabecera)

    chat = gr.ChatInterface(
        fn=responder_chat,
        chatbot=gr.Chatbot(height=520, show_label=False),
        textbox=gr.Textbox(placeholder="Ej: Hola, soy de Ingeniería en Informática...", container=False, scale=7)
    )

print("🚀 Generando enlace público de la interfaz final unificada...")
portal_simulado.launch(share=True, css=estilos_css)

⏳ Cargando base de datos y configurando sistema...
✅ Excel de horarios cargado correctamente.
✅ Mallas curriculares procesadas.

🚀 Generando enlace público de la interfaz final unificada...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://44fbd6c40b98e05126.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
